In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

# Identify the project root
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {processed_data_path}"
    )

df = pd.read_csv(processed_data_path)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found."
    )

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Full dataset shape:", df.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Training missing values:", int(X_train.isna().sum().sum()))
print("Testing missing values:", int(X_test.isna().sum().sum()))

class_distribution = pd.DataFrame({
    "Class": [0, 1],
    "Training Count": [
        int((y_train == 0).sum()),
        int((y_train == 1).sum())
    ],
    "Testing Count": [
        int((y_test == 0).sum()),
        int((y_test == 1).sum())
    ]
})

class_distribution

Full dataset shape: (4238, 16)
X_train shape: (3390, 15)
X_test shape: (848, 15)
y_train shape: (3390,)
y_test shape: (848,)
Training missing values: 521
Testing missing values: 124


,Class,Training Count,Testing Count
0,0,2875,719
1,1,515,129


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

continuous_features = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

binary_features = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

categorical_features = [
    "gender",
    "education"
]

all_defined_features = (
    continuous_features
    + binary_features
    + categorical_features
)

missing_features = sorted(
    set(X_train.columns) - set(all_defined_features)
)

unexpected_features = sorted(
    set(all_defined_features) - set(X_train.columns)
)

if missing_features:
    raise ValueError(
        f"Features not assigned to a group: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Defined features not found in the dataset: "
        f"{unexpected_features}"
    )

continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_pipeline,
            continuous_features
        ),
        (
            "binary",
            binary_pipeline,
            binary_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Continuous features:", len(continuous_features))
print("Binary features:", len(binary_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(all_defined_features))

Continuous features: 8
Binary features: 5
Categorical features: 2
Total features: 15


In [3]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            logistic_model
        )
    ]
)

print(logistic_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('continuous',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'cigarettes_per_day',
                                                   'total_cholesterol',
                                                   'systolic_bp',
                                                   'diastolic_bp', 'bmi',
                                                   'heart_rate', 'glucose']),
                                                 ('binary',
                                                  Pipeline(steps=[('imputer',
                                                  

In [4]:
import time

training_start = time.perf_counter()

logistic_pipeline.fit(
    X_train,
    y_train
)

training_end = time.perf_counter()
training_time = training_end - training_start

fitted_classifier = (
    logistic_pipeline
    .named_steps["classifier"]
)

print(f"Training time: {training_time:.4f} seconds")
print("Model classes:", fitted_classifier.classes_.tolist())
print("Iterations used:", fitted_classifier.n_iter_.tolist())

Training time: 0.0719 seconds
Model classes: [0, 1]
Iterations used: [46]
